In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")
login(token = secret_value_0)
print(":done")


In [ ]:
!mkdir -p /kaggle/working/fd_llm_output
!cp -r /kaggle/input/datasets/bhavyranka/800-zip/* /kaggle/working/fd_llm_output/

In [ ]:
!pip install -U bitsandbytes peft
!install transformers==4.38.0

In [ ]:
import os
import json
import random
import logging
from collections import Counter
 
import numpy as np
import torch
 
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
 
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


In [ ]:
DATA_PATH = "/kaggle/input/datasets/bhavyranka/fd-llm-crwu-dataset/cwru_stat_dataset.json"
DATA_TYPE = "stat"                        # "stat" | "fft"
 
# Base model (same one you fine-tuned from)
MODEL_ID  = "meta-llama/Meta-Llama-3-8B-Instruct"
 
# Path to your saved checkpoint — change to whichever checkpoint you want
CHECKPOINT_DIR = "/kaggle/working/fd_llm_output"
 
LOAD_IN_4BIT = True
MAX_SEQ_LEN  = 1024
SEED         = 42
TEST_SIZE    = 0.10     # same split as training so test set is identical
 
OUTPUT_DIR   = "/kaggle/working/fd_llm_eval"


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)
 
FAULT_LABELS = ["NO", "IRF", "ORF", "REF"]
LABEL2ID     = {l: i for i, l in enumerate(FAULT_LABELS)}
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
logger.info(f"Loading data from {DATA_PATH}")
with open(DATA_PATH, "r") as f:
    data = json.load(f)
logger.info(f"Total samples: {len(data)}")
logger.info(f"Label distribution [full]: {dict(Counter(d['output'] for d in data))}")
 
_, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=[d["output"] for d in data]
)
logger.info(f"Test samples (held-out 10%): {len(test_data)}")
logger.info(f"Label distribution [test]: {dict(Counter(d['output'] for d in test_data))}")


In [ ]:
logger.info(f"Loading base model: {MODEL_ID}")
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
 
bnb_config = None
if LOAD_IN_4BIT:
    try:
        import bitsandbytes
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        logger.info("4-bit quantisation enabled.")
    except ImportError:
        logger.warning("bitsandbytes not found — loading in bfloat16.")
 
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if bnb_config is None else None,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False


In [ ]:
logger.info(f"Loading LoRA checkpoint from: {CHECKPOINT_DIR}")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model.eval()
logger.info("Model ready for inference.")


In [ ]:
def build_inference_prompt(instruction: str, input_text: str) -> str:
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )
 
def extract_label(generated_text: str) -> str:
    text = generated_text.strip().upper()
    for label in FAULT_LABELS:
        if label in text:
            return label
    first_word = text.split()[0] if text else "UNKNOWN"
    return first_word
 
@torch.no_grad()
def run_inference(samples, batch_size: int = 8) -> list:
    predictions = []
 
    for i in range(0, len(samples), batch_size):
        batch   = samples[i: i + batch_size]
        prompts = [build_inference_prompt(s["instruction"], s["input"]) for s in batch]
 
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
 
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
 
        # Decode only the newly generated tokens
        generated = outputs[:, inputs["input_ids"].shape[1]:]
        decoded   = tokenizer.batch_decode(generated, skip_special_tokens=True)
        predictions.extend([extract_label(d) for d in decoded])
 
        logger.info(f"  Inference: {min(i + batch_size, len(samples))}/{len(samples)}")
 
    return predictions
 
logger.info("Running inference on test set ...")
preds  = run_inference(test_data)
truths = [d["output"] for d in test_data]


In [ ]:
preds_clean = [p if p in FAULT_LABELS else "__INVALID__" for p in preds]
 
n_invalid = sum(1 for p in preds_clean if p == "__INVALID__")
if n_invalid:
    logger.warning(f"{n_invalid} predictions were unmappable — counted as wrong.")
 
acc  = accuracy_score(truths, preds_clean)
prec = precision_score(truths, preds_clean, labels=FAULT_LABELS, average="weighted", zero_division=0)
rec  = recall_score(truths, preds_clean, labels=FAULT_LABELS, average="weighted", zero_division=0)
f1   = f1_score(truths, preds_clean, labels=FAULT_LABELS, average="weighted", zero_division=0)
cm   = confusion_matrix(truths, preds_clean, labels=FAULT_LABELS)
 
print("\n" + "=" * 60)
print(f"  CHECKPOINT : {CHECKPOINT_DIR}")
print(f"  DATA TYPE  : {DATA_TYPE.upper()}")
print(f"  TEST SIZE  : {len(test_data)} samples")
print("=" * 60)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("\n  Classification Report:")
print(classification_report(truths, preds_clean, labels=FAULT_LABELS, zero_division=0))
print("  Confusion Matrix:")
header = "       " + "  ".join(f"{l:>5}" for l in FAULT_LABELS)
print(header)
for i, row in enumerate(cm):
    print(f"  {FAULT_LABELS[i]:>5}  {'  '.join(f'{v:5d}' for v in row)}")
print("=" * 60 + "\n")


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
results = {
    "checkpoint":       CHECKPOINT_DIR,
    "data_type":        DATA_TYPE,
    "test_samples":     len(test_data),
    "accuracy":         acc,
    "precision":        prec,
    "recall":           rec,
    "f1":               f1,
    "confusion_matrix": cm.tolist(),
    "invalid_preds":    n_invalid,
}
 
results_path = os.path.join(OUTPUT_DIR, f"1eval_{DATA_TYPE}_checkpoint800.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
logger.info(f"Results saved to {results_path}")
